In [1]:
import os
import yaml
from tjmonopix2.system.bdaq53 import BDAQ53
from tjmonopix2.system.tjmonopix2 import TJMonoPix2

In [2]:
# Initialization
with open(os.path.join('..', 'tjmonopix2', 'system', 'bdaq53.yaml'), 'r') as f:
    cnfg = yaml.full_load(f)

daq = BDAQ53(cnfg)
daq.init()

2026-08-25 12:47:34,834 [TJ-Monopix2      ] - SUCCESS Found board BDAQ53 running firmware version 0.2
2026-08-25 12:47:34,961 [basil.HL.si570   ] - INFO    Changed Si570 reference frequency to 160.0 MHz


In [5]:
daq['DAQ_CONTROL']['MPXOBX_ENABLE'] = 0
daq['DAQ_CONTROL']['MPXOBX_RESET'] = 0

daq['DAQ_CONTROL'].write()

In [3]:
from tjmonopix2.system.adg72_tca955 import tca9555

tca9555_conf = {'name': 'tca9555', 'type': 'tca9555', 'interface': 'intf', 'base_addr': 0x40}
chip_en = tca9555(daq['i2c'], tca9555_conf)
chip_en.init()

In [6]:
daq['i2c'].scan_i2c_address()

Found I2C at basil address:    0x40
Corresponding to I2C address:  0x20 , 0b100000
-----------------------------------------------
Found I2C at basil address:    0x98
Corresponding to I2C address:  0x4c , 0b1001100
-----------------------------------------------
Found I2C at basil address:    0xa2
Corresponding to I2C address:  0x51 , 0b1010001
-----------------------------------------------
Found I2C at basil address:    0xba
Corresponding to I2C address:  0x5d , 0b1011101
-----------------------------------------------
Found I2C at basil address:    0xc8
Corresponding to I2C address:  0x64 , 0b1100100
-----------------------------------------------


['0b100000', '0b1001100', '0b1010001', '0b1011101', '0b1100100']

In [9]:
chip_en.enable_pins([0, 6, 3, 2, 1])

In [10]:
chip_en.disable_all_pins()

In [8]:
chip = TJMonoPix2(daq, chip_id=16)
chip.init()

2026-08-17 14:17:32,677 [TJ-Monopix2 - W00R00] - WARNING No explicit configuration supplied. Using 'default.cfg.yaml'!
2026-08-17 14:17:32,701 [TJ-Monopix2 - W00R00] - INFO    Initializing communication...
2026-08-17 14:17:32,704 [TJ-Monopix2 - W00R00] - SUCCESS Communication established


In [6]:
# chip.masks['enable'][0:512, 0:512] = False
for i in range(16):
    chip._write_register(171 + i, 0x0000)

In [9]:
# Testing
chip.registers["ITHR"].write(50)  # Write registers using register object
chip._write_register(151, 25)  # Write registers directly

chip.masks["enable"][:2, :] = True  # Usage of masks as 2d-array with regular python indexing
chip.masks.update()

daq.rx_channels["rx0"].set_en(True)  # Enable RX module in firmware

daq.reset_fifo()  # Clear FIFO before reading data
raw_data = daq["FIFO"].get_data()  # Get FIFO contents
hit_data, reg_data = chip.interpret_data(raw_data)  # Interpret data

print(chip.registers["ITHR"].read())  # Read registers using register object
print(chip._get_register_value(151))  # Read registers (almost) directly

2026-08-17 14:17:41,067 [TJ-Monopix2 - W00R00] - WARNING Timeout while waiting for register response.
2026-08-17 14:17:41,534 [TJ-Monopix2 - W00R00] - WARNING Timeout while waiting for register response.
2026-08-17 14:17:42,003 [TJ-Monopix2 - W00R00] - WARNING Timeout while waiting for register response.
2026-08-17 14:17:42,464 [TJ-Monopix2 - W00R00] - WARNING Timeout while waiting for register response.
2026-08-17 14:17:42,929 [TJ-Monopix2 - W00R00] - WARNING Timeout while waiting for register response.
2026-08-17 14:17:43,397 [TJ-Monopix2 - W00R00] - WARNING Timeout while waiting for register response.
2026-08-17 14:17:43,865 [TJ-Monopix2 - W00R00] - WARNING Timeout while waiting for register response.
2026-08-17 14:17:44,326 [TJ-Monopix2 - W00R00] - WARNING Timeout while waiting for register response.
2026-08-17 14:17:44,797 [TJ-Monopix2 - W00R00] - WARNING Timeout while waiting for register response.
2026-08-17 14:17:45,263 [TJ-Monopix2 - W00R00] - WARNING Timeout while waiting for

RuntimeError: Timeout while waiting for register response.

In [8]:
# Closing
daq.close()